# Spatial Color Affinity Loss

Trains the colorization model with an additional Spatial Color Affinity loss on top
of the standard L1 + GAN baseline.

The idea is simple: adjacent pixels with similar luminance are likely part of the
same surface, so they should receive similar colors. The loss penalizes cases where
the generator violates this consistency relative to the ground truth. Unlike TV loss,
it only fires when luminance similarity suggests the colors should agree — so it does
not suppress legitimate color boundaries.

This experiment reuses the pretrained generator from Stage 1 of the baseline run.


In [ ]:
!pip install fastai>=2.7 scikit-image tqdm -q


In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb

import torch
from torch import nn, optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

from fastai.vision.learner import create_body
from torchvision.models.resnet import resnet18
from fastai.vision.models.unet import DynamicUnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
SIZE = 256
BASELINE_CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints"


def build_res_unet(n_input=1, n_output=2, size=256, dropout=0.5):
    try:
        from torchvision.models import ResNet18_Weights
        model = resnet18(weights='DEFAULT')
    except:
        try:
            model = resnet18(pretrained=True)
        except:
            model = resnet18(pretrained=False)
            print("warning: could not load pretrained weights")

    if n_input == 1:
        old_conv = model.conv1
        with torch.no_grad():
            new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        model.conv1 = new_conv

    body = create_body(model, cut=-2)
    return DynamicUnet(body, n_output, (size, size)).to(
        torch.device("cuda" if torch.cuda.is_available() else "cpu"))


class PatchDiscriminator(nn.Module):
    def __init__(self, input_c, num_filters=64, n_down=3):
        super().__init__()
        self.model = self.get_layers(input_c, num_filters, n_down)

    def get_layers(self, input_c, num_filters, n_down):
        model = [self.get_conv(input_c, num_filters, norm=False)]
        for i in range(n_down):
            model += [self.get_conv(num_filters * 2**i, num_filters * 2**(i+1),
                                    stride=1 if i == (n_down - 1) else 2)]
        model += [self.get_conv(num_filters * 2**n_down, 1, stride=1, norm=False, act=False)]
        return nn.Sequential(*model)

    def get_conv(self, in_c, out_c, kernel_size=4, stride=2, padding=1, norm=True, act=True):
        layers = [nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=not norm)]
        if norm: layers.append(nn.BatchNorm2d(out_c))
        if act:  layers.append(nn.LeakyReLU(0.2, True))
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class GANLoss(nn.Module):
    def __init__(self, gan_mode='vanilla', real_label=1.0, fake_label=0.0):
        super().__init__()
        self.register_buffer('real_label', torch.tensor(real_label))
        self.register_buffer('fake_label', torch.tensor(fake_label))
        self.loss = nn.BCEWithLogitsLoss() if gan_mode == 'vanilla' else nn.MSELoss()

    def get_labels(self, preds, target_is_real):
        labels = self.real_label if target_is_real else self.fake_label
        return labels.expand_as(preds)

    def __call__(self, preds, target_is_real):
        return self.loss(preds, self.get_labels(preds, target_is_real))


def init_weights(net, gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and 'Conv' in classname:
            nn.init.normal_(m.weight.data, mean=0.0, std=gain)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'BatchNorm2d' in classname:
            nn.init.normal_(m.weight.data, 1., gain)
            nn.init.constant_(m.bias.data, 0.)
    net.apply(init_func)
    return net


def init_model(model, device):
    return init_weights(model.to(device))


def total_variation_loss(img):
    diff_h = torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:])
    diff_v = torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :])
    return diff_h.mean() + diff_v.mean()


def contrast_loss(fake_img, real_img):
    fake_L = fake_img[:, 0:1, :, :]
    real_L = real_img[:, 0:1, :, :]
    fake_std = torch.std(fake_L.view(fake_L.size(0), -1), dim=1)
    real_std = torch.std(real_L.view(real_L.size(0), -1), dim=1)
    return torch.abs(fake_std - real_std).mean()


class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.count, self.avg, self.sum = [0.] * 3
    def update(self, val, count=1):
        self.count += count
        self.sum  += count * val
        self.avg   = self.sum / self.count


def lab_to_rgb(L, ab):
    L  = (L + 1.) * 50.
    ab = ab * 110.
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    return np.stack([lab2rgb(img) for img in Lab], axis=0)


def visualize(model, data, save=False, path="result.png"):
    model.net_G.eval()
    with torch.no_grad():
        model.setup_input(data)
        model.forward()
    model.net_G.train()
    fake = lab_to_rgb(model.L, model.fake_color.detach())
    real = lab_to_rgb(model.L, model.ab)
    n = min(5, len(model.L))
    fig, axes = plt.subplots(3, n, figsize=(3*n, 9))
    for i in range(n):
        axes[0, i].imshow(model.L[i][0].cpu(), cmap='gray'); axes[0, i].axis('off')
        axes[1, i].imshow(fake[i]);                           axes[1, i].axis('off')
        axes[2, i].imshow(real[i]);                           axes[2, i].axis('off')
    axes[0, 0].set_ylabel("input");       axes[1, 0].set_ylabel("output")
    axes[2, 0].set_ylabel("ground truth")
    plt.tight_layout()
    if save: plt.savefig(path, bbox_inches='tight')
    plt.show()


print("definitions loaded")


In [ ]:
class SpatialColorAffinityLoss(nn.Module):
    def __init__(self, threshold=0.1):
        super().__init__()
        self.threshold = threshold

    def forward(self, L, ab_pred, ab_target):
        L_diff_h = torch.abs(L[:, :, :, :-1]  - L[:, :, :, 1:])
        L_diff_v = torch.abs(L[:, :, :-1, :]  - L[:, :, 1:, :])

        similar_h = (L_diff_h < self.threshold).float()
        similar_v = (L_diff_v < self.threshold).float()

        ab_diff_pred_h = torch.abs(ab_pred[:, :, :, :-1]   - ab_pred[:, :, :, 1:]).mean(dim=1, keepdim=True)
        ab_diff_pred_v = torch.abs(ab_pred[:, :, :-1, :]   - ab_pred[:, :, 1:, :]).mean(dim=1, keepdim=True)
        ab_diff_gt_h   = torch.abs(ab_target[:, :, :, :-1] - ab_target[:, :, :, 1:]).mean(dim=1, keepdim=True)
        ab_diff_gt_v   = torch.abs(ab_target[:, :, :-1, :] - ab_target[:, :, 1:, :]).mean(dim=1, keepdim=True)

        loss_h = (similar_h * torch.abs(ab_diff_pred_h - ab_diff_gt_h)).mean()
        loss_v = (similar_v * torch.abs(ab_diff_pred_v - ab_diff_gt_v)).mean()
        return loss_h + loss_v


In [ ]:
class MainModel(nn.Module):
    def __init__(self, net_G=None, lr_G=2e-4, lr_D=2e-4,
                 beta1=0.5, beta2=0.999, lambda_L1=100., lambda_affinity=1.0):
        super().__init__()
        self.device = device
        self.lambda_L1       = lambda_L1
        self.lambda_affinity = lambda_affinity

        self.net_G = net_G.to(self.device) if net_G else                      init_model(build_res_unet(n_input=1, n_output=2, size=SIZE), self.device)
        self.net_D         = init_model(PatchDiscriminator(input_c=3, n_down=3, num_filters=64), self.device)
        self.GANcriterion  = GANLoss(gan_mode='vanilla').to(self.device)
        self.L1criterion   = nn.L1Loss()
        self.aff_criterion = SpatialColorAffinityLoss(threshold=0.1)
        self.opt_G = optim.Adam(self.net_G.parameters(), lr=lr_G, betas=(beta1, beta2))
        self.opt_D = optim.Adam(self.net_D.parameters(), lr=lr_D, betas=(beta1, beta2))

    def set_requires_grad(self, model, requires_grad=True):
        for p in model.parameters():
            p.requires_grad = requires_grad

    def setup_input(self, data):
        self.L  = data['L'].to(self.device)
        self.ab = data['ab'].to(self.device)

    def forward(self):
        self.fake_color = self.net_G(self.L)

    def backward_D(self):
        fake_preds = self.net_D(torch.cat([self.L, self.fake_color], dim=1).detach())
        self.loss_D_fake = self.GANcriterion(fake_preds, False)
        real_preds = self.net_D(torch.cat([self.L, self.ab], dim=1))
        self.loss_D_real = self.GANcriterion(real_preds, True)
        self.loss_D = (self.loss_D_fake + self.loss_D_real) * 0.5
        self.loss_D.backward()

    def backward_G(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        fake_preds = self.net_D(fake_image)
        self.loss_G_GAN      = self.GANcriterion(fake_preds, True)
        self.loss_G_L1       = self.L1criterion(self.fake_color, self.ab) * self.lambda_L1
        self.loss_G_affinity = self.aff_criterion(self.L, self.fake_color, self.ab) * self.lambda_affinity
        self.loss_G = self.loss_G_GAN + self.loss_G_L1 + self.loss_G_affinity
        self.loss_G.backward()

    def optimize(self):
        self.forward()
        self.net_D.train(); self.set_requires_grad(self.net_D, True)
        self.opt_D.zero_grad(); self.backward_D(); self.opt_D.step()
        self.net_G.train(); self.set_requires_grad(self.net_D, False)
        self.opt_G.zero_grad(); self.backward_G(); self.opt_G.step()


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/datasets/coco_subset_16000"
NUM_IMAGES   = 13000


class ColorizationDataset(Dataset):
    def __init__(self, paths, split='train'):
        self.paths = paths
        self.split = split
        if split == 'train':
            self.transforms = transforms.Compose([
                transforms.Resize((256, 256), Image.BICUBIC),
                transforms.RandomHorizontalFlip(),
            ])
        else:
            self.transforms = transforms.Resize((256, 256), Image.BICUBIC)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transforms(img)
        img_lab = rgb2lab(np.array(img)).astype("float32")
        img_lab = transforms.ToTensor()(img_lab)
        L  = img_lab[[0], ...] / 50. - 1.
        ab = img_lab[[1, 2], ...] / 110.
        return {'L': L, 'ab': ab}

    def __len__(self):
        return len(self.paths)


def make_dataloaders(paths, split='train', batch_size=16, n_workers=2):
    return DataLoader(ColorizationDataset(paths, split), batch_size=batch_size,
                      num_workers=n_workers, pin_memory=True, shuffle=(split == 'train'))


# load paths
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
paths = []
for ext in image_extensions:
    paths.extend(glob.glob(os.path.join(DATASET_PATH, ext)))
    paths.extend(glob.glob(os.path.join(DATASET_PATH, '**', ext), recursive=True))

if not paths:
    raise RuntimeError(f"no images found in {DATASET_PATH}")

np.random.seed(123)
if len(paths) > NUM_IMAGES:
    paths = np.random.choice(paths, NUM_IMAGES, replace=False)

rand_idxs   = np.random.permutation(len(paths))
train_paths = paths[rand_idxs[:int(len(paths) * 0.8)]]
val_paths   = paths[rand_idxs[int(len(paths) * 0.8):]]

BATCH_SIZE  = 16
NUM_WORKERS = 2

train_dl = make_dataloaders(train_paths, split='train', batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)
val_dl   = make_dataloaders(val_paths,   split='val',   batch_size=BATCH_SIZE, n_workers=NUM_WORKERS)

print(f"train: {len(train_paths)}  val: {len(val_paths)}")
print(f"train batches: {len(train_dl)}  val batches: {len(val_dl)}")


In [ ]:
AFFINITY_CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints_spatial_affinity"
os.makedirs(AFFINITY_CHECKPOINT_DIR, exist_ok=True)

LAMBDA_L1       = 100.0
LAMBDA_AFFINITY = 1.0

net_G = build_res_unet(n_input=1, n_output=2, size=SIZE)
pretrained = os.path.join(BASELINE_CHECKPOINT_DIR, "pretrained_generator.pth")
if os.path.exists(pretrained):
    net_G.load_state_dict(torch.load(pretrained, map_location=device))
    print(f"loaded baseline generator from {pretrained}")
else:
    print(f"baseline not found at {pretrained} - starting from scratch")


In [ ]:
model = MainModel(net_G=net_G, lambda_L1=LAMBDA_L1, lambda_affinity=LAMBDA_AFFINITY)

# resume from checkpoint if available
RESUME_TRAINING    = False
CHECKPOINT_TO_LOAD = "checkpoint_epoch_10.pth"
START_EPOCH        = 10

if RESUME_TRAINING:
    ckpt_path = os.path.join(AFFINITY_CHECKPOINT_DIR, CHECKPOINT_TO_LOAD)
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.net_G.load_state_dict(ckpt['generator_state_dict'])
        model.net_D.load_state_dict(ckpt['discriminator_state_dict'])
        model.opt_G.load_state_dict(ckpt['optimizer_G_state_dict'])
        model.opt_D.load_state_dict(ckpt['optimizer_D_state_dict'])
        print(f"resumed from epoch {ckpt['epoch']}")
    else:
        print(f"checkpoint not found: {ckpt_path}")
        RESUME_TRAINING = False


In [ ]:
GAN_EPOCHS    = 20
DISPLAY_EVERY = 200
val_data      = next(iter(val_dl))

for e in range(START_EPOCH if RESUME_TRAINING else 0, GAN_EPOCHS):
    meters = {k: AverageMeter() for k in
              ['loss_D_fake', 'loss_D_real', 'loss_D',
               'loss_G_GAN', 'loss_G_L1', 'loss_G_affinity', 'loss_G']}
    i = 0
    for data in tqdm(train_dl, desc=f"epoch {e+1}/{GAN_EPOCHS}"):
        model.setup_input(data)
        model.optimize()
        for name, meter in meters.items():
            meter.update(getattr(model, name).item(), data['L'].size(0))
        i += 1
        if i % DISPLAY_EVERY == 0:
            print(f"\nepoch {e+1}  iter {i}/{len(train_dl)}")
            for name, meter in meters.items():
                print(f"  {name}: {meter.avg:.5f}")
            visualize(model, val_data, save=True,
                      path=os.path.join(AFFINITY_CHECKPOINT_DIR, f"epoch{e+1}_iter{i}.png"))

    if (e + 1) % 5 == 0 or (e + 1) == GAN_EPOCHS:
        ckpt_path = os.path.join(AFFINITY_CHECKPOINT_DIR, f"checkpoint_epoch_{e+1}.pth")
        torch.save({
            'epoch': e + 1,
            'generator_state_dict':     model.net_G.state_dict(),
            'discriminator_state_dict': model.net_D.state_dict(),
            'optimizer_G_state_dict':   model.opt_G.state_dict(),
            'optimizer_D_state_dict':   model.opt_D.state_dict(),
        }, ckpt_path)
        print(f"  saved {ckpt_path}")

torch.save(model.net_G.state_dict(),
           os.path.join(AFFINITY_CHECKPOINT_DIR, "spatial_affinity_generator.pth"))
print("done")


In [ ]:
visualize(model, next(iter(val_dl)), save=True,
          path=os.path.join(AFFINITY_CHECKPOINT_DIR, "final_results.png"))
